In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/llm-post-training'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Working in: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working in: /content/drive/MyDrive/llm-post-training


In [2]:
# First time load
# !git clone https://github.com/psaesha/LLM-Post-Training.git repo
%cd repo
!ls
!git pull

/content/drive/MyDrive/llm-post-training/repo
LICENSE  notebooks  pyproject.toml  README.md  uv.lock
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 1), reused 4 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 792 bytes | 18.00 KiB/s, done.
From https://github.com/psaesha/LLM-Post-Training
   6a1e87c..828e3a4  main       -> origin/main
Updating 6a1e87c..828e3a4
Fast-forward
 configs/sft.yaml | 47 +++++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 47 insertions(+)
 create mode 100644 configs/sft.yaml


In [3]:
!uv sync

Resolved 50 packages in 5ms
Checked 48 packages in 116ms


In [4]:
from huggingface_hub import login

login()

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# memory check
print(f"Model loaded.")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
print(f"Model dtype: {next(model.parameters()).dtype}")
print(f"Param count: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model loaded.
GPU memory used: 1.17 GB
GPU memory reserved: 3.11 GB
Model dtype: torch.bfloat16
Param count: 0.89B


In [6]:
def generate(prompt, max_new_tokens=150):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

# Test prompts — what's the base model's Hindi like?
test_prompts = [
    "भारत की राजधानी क्या है?",
    "मुझे एक छोटी कहानी सुनाओ।",
    "मशीन लर्निंग क्या है? सरल भाषा में समझाओ।",
    "Hindi mein ek chhoti kavita likho.",  # code-mixed
]

for p in test_prompts:
    print(f"PROMPT: {p}")
    print(f"OUTPUT: {generate(p)}")
    print("-" * 80)

PROMPT: भारत की राजधानी क्या है?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


OUTPUT:  राजधानी बार्षिया है।
--------------------------------------------------------------------------------
PROMPT: मुझे एक छोटी कहानी सुनाओ।
OUTPUT:  बहुत अच्छी है। मुझे एक छोटी कहानी सुनाएं। बहुत अच्छी है। मुझे एक छोटी कहानी सुनाएं। बहुत अच्छी है। मुझे एक छोटी कहानी सुनाएं। बहुत अच्छी है। मुझे एक छोटी कहानी स
--------------------------------------------------------------------------------
PROMPT: मशीन लर्निंग क्या है? सरल भाषा में समझाओ।
OUTPUT:  निचे दिए गए अध्ययन सूचना के आधार पर जो विषय पर आपको लगभग तीन मिनट बिलकुल सक्षम होनी चाहिए: विद्यालय अध्यापक, शिक्षण विज्ञान विज्ञान, प्रौढ़ा सांख्यिकी और अन
--------------------------------------------------------------------------------
PROMPT: Hindi mein ek chhoti kavita likho.
OUTPUT:  Kya mera namee na ho. Mera namee na ho. Kya mera namee na ho. Mera namee na ho. Kya mera namee na ho. Mera namee na ho. Kya mera namee na ho. Mera namee na ho. Kya mera namee na ho. Mera namee na ho. Kya mera namee na ho. Mera namee na ho. Kya mera namee

In [8]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 32.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 27.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 191.2 MB/s  0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [trl]


In [12]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# tiny toy dataset — 20 examples, just to confirm everything wires up
toy_data = [
    {"messages": [
        {"role": "user", "content": "भारत की राजधानी क्या है?"},
        {"role": "assistant", "content": "भारत की राजधानी नई दिल्ली है।"}
    ]},
    {"messages": [
        {"role": "user", "content": "2 + 2 क्या है?"},
        {"role": "assistant", "content": "2 + 2 = 4 होता है।"}
    ]},
    # add ~18 more similar pairs, or just multiply this list
] * 10

dataset = Dataset.from_list(toy_data)

# prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # should show ~1% of params trainable

training_args = SFTConfig(
    output_dir="/content/toy_sft",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=5,
    max_steps=50,  # just 50 steps to confirm loss goes down
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
    save_strategy="no",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


Tokenizing train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,2.519958
10,1.154743
15,0.820150
20,0.728607
25,0.700224
30,0.702452
35,0.685038
40,0.682375
45,0.683616
50,0.688760


TrainOutput(global_step=50, training_loss=0.936592230796814, metrics={'train_runtime': 63.0415, 'train_samples_per_second': 3.173, 'train_steps_per_second': 0.793, 'total_flos': 104045460940800.0, 'train_loss': 0.936592230796814, 'epoch': 10.0})

In [13]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
